## Step 1. Import the libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import string
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
import os
from textblob import TextBlob
from nltk.stem import PorterStemmer
from textblob import Word
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
import sklearn.feature_extraction.text as text
from sklearn import model_selection,preprocessing,linear_model,naive_bayes,metrics,svm
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score
from io import StringIO
import seaborn as sns
import pandas.util.testing as tm

## Step 2. Import Data

In [ ]:
Data = pd.read_csv(r"D:\Practice\ArtificialIntelligence\Natural Language Process\Industry Applications\Multi_Classify(consumer_complaint_classify)\consumer_complaints.csv",
                  encoding='latin-1')

## Step 3. Data understanding

In [ ]:
Data.dtypes

In [ ]:
Data.head()

In [ ]:
# Selecting required columns and rows
Data = Data[['product','consumer_complaint_narrative']]
Data = Data[pd.notnull(Data['consumer_complaint_narrative'])]
Data.head()

In [ ]:
#Factorizing the category column
Data['category_id'] = Data['product'].factorize()[0]
Data.head()


In [ ]:
# Check the distribution of complaints by category
Data.groupby('product').consumer_complaint_narrative.count()

In [ ]:
#Lets plot it and see
fig = plt.figure(figsize=(8,6))
Data.groupby('product').consumer_complaint_narrative.count().plot.bar(ylim=0)
plt.show()

## Step 4. Splitting the data

In [ ]:
#Split the data into train and validation
train_x,valid_x,train_y,valid_y = model_selection.train_test_split(Data['consumer_complaint_narrative'],Data['product'])

## Step 5. Feature engineering using TF-IDF

In [ ]:
# Consider maximum features to be 5000
encoder = preprocessing.LabelEncoder()
train_y = encoder.fit_transform(train_y)
valid_y = encoder.fit_transform(valid_y)

In [ ]:
tfidf_vect = TfidfVectorizer(analyzer='word',token_pattern=r'\w{1,}',max_features=5000)
tfidf_vect.fit(Data['consumer_complaint_narrative'])
xtrain_tfidf = tfidf_vect.transform(train_x)
xvalid_tfidf = tfidf_vect.transform(valid_x)

In [ ]:
xtrain_tfidf

In [ ]:
len(train_y)

## Step 6. Model Building and Evaluation

In [ ]:
model = linear_model.LogisticRegression().fit(xtrain_tfidf,train_y)

In [ ]:
#Checking accuracy
accuracy = metrics.accuracy_score(model.predict(xvalid_tfidf),valid_y)
print("Accuracy: ",accuracy)

In [ ]:
#Calssification report
print(metrics.classification_report(valid_y,model.predict(xvalid_tfidf),
                                   target_names=Data['product'].unique()))

In [ ]:
#confusion matrix
from sklearn.metrics import confusion_matrix
conf_mat = confusion_matrix(valid_y,model.predict(xvalid_tfidf))

In [ ]:
#visualizing the confusion matrix
category_id_df = Data[['product','category_id']].drop_duplicates().sort_values('category_id')
category_to_id = dict(category_id_df.values)
id_to_category = dict(category_id_df[['category_id','product']].values)

fig,ax = plt.subplots(figsize=(8,6))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap="BuPu",
    xticklabels=category_id_df[['product']].values,yticklabels=category_id_df[['product']].values)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
#Prediction example
texts = ["I have an open and current mortgage with Chase Bank # XXXX. Chase is reporting the loan payments to XXXX but XXXX is surpressing the information and reporting the loan as Discharged in BK. This mortgage was reaffirmed in a Chapter XXXX BK discharged dated XXXX/XXXX/2013. Chase keeps referring to BK Law for Chapter XXXX and we keep providing documentation for Chapter XXXX, and the account should be open and current with all the payments  "]
text_features = tfidf_vect.transform(texts)
predictions = model.predict(text_features)

In [ ]:
print(texts)
print(f"-predicted as '{id_to_category[predictions[0]]}'")